# Experiment 1: summarize one approach
CPU-only analysis of completed recordings. Edit the configuration below, then Run All. A single checkpoint is required; mixed protocols/checkpoints fail validation. Evaluation seeds are not training seeds.

In [ ]:
import os, json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Find this checkout whether Jupyter starts in the repository or notebooks directory.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "notebooks" / "exp1_analysis.py").exists())
sys.path.insert(0, str(REPO / "notebooks"))
import exp1_analysis as exp1
# Refresh helper changes even when this Jupyter kernel was already running.
import importlib
exp1 = importlib.reload(exp1)

CONFIG = {
    "input_roots": [str(REPO / "evaluation/exp1/b1z1_ppo_pos")],
    "task": "b1z1_ppo_pos",
    "checkpoint_sha256": None,  # Optional exact hash from metadata.json; otherwise require a unique hash.
    "display_name": "Pos",
    "expected_seeds": [0, 1, 2],
    "output_dir": str(REPO / "analysis/exp1/b1z1_ppo_pos"),
    "histogram_edges": np.linspace(-0.5, 0.5, 101).tolist(),  # Metres; reuse across approaches.
    "histogram_frame": "yaw",  # Successor base-yaw frame; "world" is also supported.
    "include_combined": True,  # Explicitly opt in to pooled stationary+locomotion trials.
    "acceleration_metrics": True,  # Backward differences, no filtering, no reset/gap crossings.
}
# Optional JSON override for reproducible batch execution/testing.
if os.environ.get("EXP1_ANALYSIS_CONFIG"):
    CONFIG.update(json.loads(Path(os.environ["EXP1_ANALYSIS_CONFIG"]).read_text()))
CONFIG


## Definitions
Event recovery is also evaluated separately for EE-position tolerance and for planar velocity plus yaw-rate tolerances. The eligible-event denominator, response window, contiguous dwell and whole-trial failure censoring are unchanged. Per-event outcomes and per-seed recovered counts are saved; each seed's recovery fraction is averaged equally across seeds. Regenerate summaries to add these metrics.
The additional `S_track_velocity` requires planar velocity **and** yaw-rate tolerances, while `S_track_manipulation` requires only the EE-position tolerance. Each uses the full prescribed horizon, with zero credit for failed/missing steps and only its own errors required finite. These are computed from transitions, then averaged equally across trials and evaluation seeds. Rerun this notebook for all methods before comparing the new scores.
Tracking errors use finite pre-failure samples. Each trial contributes equally within its evaluation seed; seeds then contribute equally. `S_track` uses the full prescribed horizon, including zero credit after failure. SD is the sample SD (`ddof=1`); one seed has undefined SD.

Physical-limit percentages pool valid joint-substep numerators/denominators within each seed. Event peaks and integrals require complete response windows. Any physical failure censors all recovery events in that trial, matching the evaluator. Recovery times are averaged only over recovered events; no missing value becomes zero.

In [ ]:
summary = exp1.summarize(CONFIG)
display(pd.DataFrame(summary["aggregates"]))
print("\n".join(summary["diagnostics"]) or "No missing-data or online cross-check warnings.")
print("Exported:", CONFIG["output_dir"])


## Portable exports
Exact equal-seed weighted box statistics (5/25/50/75/95 percentiles) are also exported in `summary.json`. They include all valid errors, including histogram tails. Rerun this notebook for older summaries that lack `ee_boxplots`.
`summary.json` contains provenance, protocols, per-trial/event results, per-seed values, aggregate statistics and diagnostics. `summary.csv` contains aggregate rows; `ee_histograms.csv` contains per-seed counts and explicit underflow/overflow denominators. `COMPLETE` is written only after all exports succeed.

Histograms use signed measured-minus-reference errors, in metres, rotated with the matching successor orientation. Counts are normalized per seed using **all valid samples**, including tails, before seed averaging. Failed tails are never padded.

In [ ]:
histograms = pd.read_csv(Path(CONFIG["output_dir"]) / "ee_histograms.csv")
tails = histograms.drop_duplicates(["seed", "condition", "load_range", "motion", "axis"])
display(tails[["seed", "condition", "load_range", "motion", "axis", "valid_samples", "underflow", "overflow"]])
